# 03. Generative Language Models

## Paper requirements

3 step of the reproduction of a DNN Scientific Paper(SP):
1. Choose a SP that needs to be published after 1st of January 2023
2. Do a resume of the main parts of the paper
3. Reproduce in details part of the paper code

You can do it all at once, but strongly recommended to do it on stages.

## GELU (Gaussian Error Linear Units)

Problem wtih RELU. If the activation functions is negative it outputs 0, if it outputs 0 then the gradient is 0, then the unit wont be doing any learning on that iteration.

If you notice that the DNN is learning really slow and you are using RELU, that could be the problem, you might have many units that are not learning due to negative activation going to 0.\
Quick fix is to use leaky_relu that puts a 0,0001 instead of 0.

<img src="Images/gelu.png" width=500>

(Gemini)\
**GELU** is a high-performance, non-linear activation function commonly used in Transformer architectures like **BERT** and **GPT-2**.

#### 1. The Mathematical Definition
The function is defined as:
$$GELU(x) = x\Phi(x)$$

Where:
* $x$ is the input to the activation.
* $\Phi(x)$ is the **Standard Gaussian Cumulative Distribution Function**.

#### 2. Probabilistic Intuition
GELU can be thought of as a smoother, probabilistic version of ReLU. Instead of a hard "gate" that shuts off at zero, it weights the input by its probability under a normal distribution:
$$x \cdot P(X \le x)$$

#### 3. Why use GELU over ReLU?
* **Smoothness:** Unlike ReLU, which has a sharp "elbow" at zero (making it non-differentiable at that exact point), GELU is smooth everywhere.
* **Vanishing Gradients:** Because GELU allows for a small, non-zero gradient even for slightly negative values, it helps mitigate the "dying ReLU" problem during backpropagation.
* **Transformer Performance:** Research has shown that the curvature of GELU allows models to converge faster and reach higher accuracy in complex NLP tasks.

> **Note:** Because the Gaussian CDF is computationally expensive to calculate exactly, most deep learning frameworks use a fast approximation:
> $$0.5x \left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715x^3)\right]\right)$$

## Root Mean Squared Propagation (RMSprop) Optimizer

RMSProp is an adaptive learning rate optimization algorithm for training deep neural networks. It tackles the vanishing/exploding gradient problem by normalizing gradients using a moving average of their squared magnitudes, allowing for faster convergence in non-stationary objectives and reducing vertical oscillations. 

In [1]:
from transformers import GPT2LMHeadModel, AutoConfig

In [3]:
config = AutoConfig.from_pretrained("gpt2")

In [4]:
gpt2 = GPT2LMHeadModel(config)

In [5]:
gpt2

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## Dropout

```
(attn_dropout): Dropout(p=0.1, inplace=False)
(resid_dropout): Dropout(p=0.1, inplace=False)
```
We can see these drop layers on top in the GPT architecture. What are they and what do they do?

Dropout is basically regularization. It is the DNN's way of doing that.

```p=0.1``` means to drop 10% of the neurons in the layers it applies to during that training step. Each training step they are different.

> Dropout is applied **only during training**, not during inference!

<img src="Images/dropout.png" width=500>

In [14]:
from keras.layers import Dense, Dropout
from keras.regularizers import L1L2, L1, L2

In [12]:
Dense(units = (20, 200), kernel_regularizer = L1L2(), bias_regularizer = L1() , activity_regularizer = L2() )

<Dense name=dense_1, built=False>

This is the regularization that we have studied in the ML course. L2 for example:

$$ J + \lambda * ||w||^2_2 $$

we add the weights of the model to the loss function with some norm

The cool thing is we can add regularization to each part of the model:
- kernal: the weights
- bias - bias
- activity - the result

This yet powerful is also dangerous. We have to very carefully tune it. The model can jump between high bias and high variance with a tiny lambda change.

It is much easier to just use a Dropout(). It is not a layer itself, it is just applied on the layer before it.

In [18]:
Dense(units = (20, 200), kernel_regularizer = L1L2(), bias_regularizer = L1() , activity_regularizer = L2() )
Dropout(rate=0.1)

<Dropout name=dropout_1, built=True>

## Norm

Normalization, what a shocker!

We want out inputs to be small numbers. It speeds up learning and computation

(Gemini)
###  Why Normalization Matters

Normalization (like **LayerNorm** in GPT-2) ensures that the values flowing through the model stay within a manageable range.

* **Learning Speed:** Smoothes the loss landscape, allowing the optimizer to reach the "bottom" (minimal error) in fewer iterations.
* **Stability:** Mitigates the **Exploding/Vanishing Gradient** problem by re-centering data at every layer.
* **Higher Learning Rates:** Because the math is more stable, we can train faster without the model "breaking."

**The Math:**
For an input $x$, we calculate the mean ($\mu$) and variance ($\sigma^2$) and transform $x$ as:
$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$
*(where $\epsilon$ is a tiny constant to prevent division by zero)*

> **Note:** GPT-2 uses **LayerNorm**, which normalizes across the hidden features of a single token, rather than **BatchNorm**, which looks at multiple sentences. This is crucial for consistent performance during inference.

### Normalization Comparison: Layer vs. Batch

| Feature | Layer Normalization (LN) | Batch Normalization (BN) |
| :--- | :--- | :--- |
| **Direction** | Across all features of **one** sample. | Across all samples of **one** feature. |
| **Batch Size** | Independent (works with Batch Size = 1). | Highly dependent (needs large batches). |
| **Use Case** | **Transformers**, RNNs, NLP. | CNNs, Computer Vision. |
| **Inference** | Same behavior as training. | Uses "running statistics" from training. |

#### 1. LayerNorm: The Transformer Choice
LayerNorm treats each sentence as its own mini-universe. It calculates the mean and variance of all 768 hidden units for a single token. This is crucial for Transformers because it ensures that the "self-attention" scores aren't affected by other unrelated sentences in the same training batch.

#### 2. BatchNorm: The CNN Classic
BatchNorm looks at "Feature #5" across 32 different images at once. While powerful for images, it fails in NLP because sentences have different lengths and meanings; forcing them to share statistics across a batch often "muddies" the specific context of a single sentence.

We use LayerNorm, because it is faster. We don't have to wait for the whole batch to be processed to apply LayerNorm, whereas with BatchNorm we do. It is just faster.

There are other normalization: LazyBatch, GroupNorm, RMSNorm etc.

In general, the model does:

1. Tokenization
2. Feature extraction (the attension block)
3. Some head (decoder) that performs a specific task

Tokenization could be a whole separate problem in itself. There are many things we can do/tweak to that process:

- Pre tokenization: Don't allow a token to contain the end of a work, then space, then the beginning of another word. First we split on space, then we tokenize.
- Remove emoticons
- Remove numbers
- Different batch sizes
- etc.

Tokenization is a different process from Attension, but they are trained together

Example theoretical questions:
- Image captioning - what is a good loss function for this problem? why choose it? What metrics can we use? Would you use attension with 2 or 15 heads?